In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

from utiles import NumbersManager

def click_by_text(driver, wait, text):
    element = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, f"label[for='{text}']"))
)

    element.click()
    return element


def fill_by_placeholder(driver, wait, placeholder, value):
    element = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, f"//*[@placeholder={repr(placeholder)}]")
        )
    )
    element.clear()
    element.send_keys(value)
    return element


def print_location(driver):
    print("URL:", driver.current_url)
    print("Title:", driver.title)

    try:
        print("Active element tag:", driver.switch_to.active_element.tag_name)
        print("Active element id:", driver.switch_to.active_element.get_attribute("id"))
        print("Active element name:", driver.switch_to.active_element.get_attribute("name"))
        print("Active element placeholder:", driver.switch_to.active_element.get_attribute("placeholder"))
    except Exception as e:
        print("Could not read active element:", e)



In [2]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 30)

driver.set_window_size(1920, 1080)
driver.get("https://www.lynkco.com/en/create-account")

# Accept cookies
wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//button[normalize-space()='Accept all cookies']")
    )
).click()






In [3]:
driver.switch_to.default_content()

iframes = driver.find_elements(By.ID, "createAccountForm")

print("matching createAccountForm iframes:", len(iframes))

target_iframe = None

for iframe in iframes:
    src = iframe.get_attribute("src") or ""
    if "login.lynkco.com" in src:
        target_iframe = iframe
        break

if target_iframe is None:
    raise Exception("Could not find the real Lynk login iframe")

driver.switch_to.frame(target_iframe)

print("Done switching to real iframe")
country_select = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "#otherCountries"))
)


Select(country_select).select_by_value("SI")

country_select = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "#countryCode"))
)


Select(country_select).select_by_value("SI")
# # Fill fields
fill_by_placeholder(driver, wait, "First name", "asdasdasd")
fill_by_placeholder(driver, wait, "Last name", "qweqweqweqwe")
fill_by_placeholder(driver, wait, "Phone number", "1")
fill_by_placeholder(driver, wait, "Email", "marwan1779724907@wshu.net")
fill_by_placeholder(driver, wait, "Password", "StrongPassword123!")



matching createAccountForm iframes: 2
Done switching to real iframe


<selenium.webdriver.remote.webelement.WebElement (session="eb0ae99aa571c0b40bad7e5db541d71e", element="f.4514977E7C9AABBAA1CA29F996A22DEE.d.EA1EBD520FE4B8B712C820DC441DE0BE.e.41")>

In [28]:
age_label = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "label[for='ageConsent_true']"))
)

age_label.click()

age_label = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "label[for='emailSubscribed_true']"))
)

age_label.click()

tos_checkbox = wait.until(
    EC.presence_of_element_located((By.ID, "tosConsent_true"))
)

if not tos_checkbox.is_selected():
    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        tos_checkbox
    )

    driver.execute_script("arguments[0].click();", tos_checkbox)

In [29]:
continue_btn = wait.until(
    EC.element_to_be_clickable((By.ID, "continue"))
)

continue_btn.click()

In [ ]:
numbers_manager = NumbersManager("database/lynk_database.db")

while True:
    number_id, number = numbers_manager.get_available_number()

    try:
        run(number)
        numbers_manager.check_number(number_id, number)

    except Exception as e:
        print(f"Error: {e}")